In [28]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import mlflow

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [29]:
# Echte Parameter — PINN soll diese finden
alpha_true = 1.0
beta_true  = 0.1
gamma_true = 1.5
delta_true = 0.075
H0, F0     = 10.0, 5.0
sigma      = 0.5

def lotka_volterra(t, z, alpha, beta, gamma, delta):
    H, F = z
    dHdt = alpha * H - beta * H * F
    dFdt = delta * H * F - gamma * F
    return [dHdt, dFdt]

t_span = (0, 15)
t_eval = np.linspace(0, 15, 100)

sol = solve_ivp(
    lotka_volterra, t_span, [H0, F0],
    args=(alpha_true, beta_true, gamma_true, delta_true),
    t_eval=t_eval
)

H_noise = sol.y[0] + np.random.normal(0, sigma, len(sol.y[0]))
F_noise = sol.y[1] + np.random.normal(0, sigma, len(sol.y[1]))

print(f"Datenpunkte: {len(sol.t)}")
print(f"H range: {sol.y[0].min():.1f} bis {sol.y[0].max():.1f}")
print(f"F range: {sol.y[1].min():.1f} bis {sol.y[1].max():.1f}")

Datenpunkte: 100
H range: 7.8 bis 40.8
F range: 3.0 bis 23.4


In [30]:
t_data = torch.tensor(sol.t,   dtype=torch.float32).reshape(-1, 1).to(device)
H_data = torch.tensor(H_noise, dtype=torch.float32).reshape(-1, 1).to(device)
F_data = torch.tensor(F_noise, dtype=torch.float32).reshape(-1, 1).to(device)

t_phys = torch.linspace(0, 15, 500).reshape(-1, 1).requires_grad_(True).to(device)

print(f"t_data shape: {t_data.shape}")
print(f"H_data shape: {H_data.shape}")
print(f"t_phys shape: {t_phys.shape}")

t_data shape: torch.Size([100, 1])
H_data shape: torch.Size([100, 1])
t_phys shape: torch.Size([500, 1])


In [31]:
class LVNet(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.l1  = nn.Linear(1, hidden)
        self.l2  = nn.Linear(hidden, hidden)
        self.l3  = nn.Linear(hidden, 2)
        self.act = nn.Tanh()
        nn.init.xavier_uniform_(self.l1.weight); nn.init.zeros_(self.l1.bias)
        nn.init.xavier_uniform_(self.l2.weight); nn.init.zeros_(self.l2.bias)
        nn.init.xavier_uniform_(self.l3.weight); nn.init.zeros_(self.l3.bias)

    def forward(self, x):
        x   = self.act(self.l1(x))
        x   = self.act(self.l2(x))
        out = self.l3(x)
        H   = out[:, 0:1]
        F   = out[:, 1:2]
        return H, F

model = LVNet().to(device)
print(f"Parameter im Netz: {sum(p.numel() for p in model.parameters())}")

Parameter im Netz: 4418


In [32]:
mse = nn.MSELoss()

def physics_loss(model, t_phys, alpha, beta, gamma, delta):
    t    = t_phys.clone().detach().requires_grad_(True).to(device)
    H, F = model(t)

    dH_dt = torch.autograd.grad(
        outputs=H, inputs=t,
        grad_outputs=torch.ones_like(H),
        create_graph=True
    )[0]
    residual_H     = dH_dt - alpha * H + beta * H * F
    physics_loss_H = mse(residual_H, torch.zeros_like(residual_H))

    dF_dt = torch.autograd.grad(
        outputs=F, inputs=t,
        grad_outputs=torch.ones_like(F),
        create_graph=True
    )[0]
    residual_F     = dF_dt - delta * H * F + gamma * F
    physics_loss_F = mse(residual_F, torch.zeros_like(residual_F))

    return physics_loss_H + physics_loss_F

print("Physics Loss Funktion definiert.")

Physics Loss Funktion definiert.


In [33]:
alpha = nn.Parameter(torch.tensor([0.5]).to(device))
beta  = nn.Parameter(torch.tensor([0.5]).to(device))
gamma = nn.Parameter(torch.tensor([0.5]).to(device))
delta = nn.Parameter(torch.tensor([0.5]).to(device))

opt = torch.optim.Adam(
    list(model.parameters()) + [alpha, beta, gamma, delta],
    lr=1e-3
)

print("Startwerte:")
print(f"  alpha={alpha.item()} (echt={alpha_true})")
print(f"  beta={beta.item()}  (echt={beta_true})")
print(f"  gamma={gamma.item()} (echt={gamma_true})")
print(f"  delta={delta.item()} (echt={delta_true})")

Startwerte:
  alpha=0.5 (echt=1.0)
  beta=0.5  (echt=0.1)
  gamma=0.5 (echt=1.5)
  delta=0.5 (echt=0.075)


In [35]:
epochs = 10000

mlflow.set_experiment("Lotka-Volterra-PINN")

with mlflow.start_run():

    mlflow.log_params({
        "epochs":             epochs,
        "lr":                 1e-3,
        "kollokationspunkte": 500,
        "sigma":              sigma,
        "hidden":             64,
        "alpha_start":        0.5,
        "beta_start":         0.5,
        "gamma_start":        0.5,
        "delta_start":        0.5
    })

    for ep in range(1, epochs + 1):
        opt.zero_grad()

        H_pred, F_pred  = model(t_data)
        data_loss_H     = mse(H_pred, H_data)
        data_loss_F     = mse(F_pred, F_data)
        physik_loss_val = physics_loss(model, t_phys, alpha, beta, gamma, delta)
        loss_total      = data_loss_H + data_loss_F + physik_loss_val

        loss_total.backward()
        opt.step()

        if ep % 10 == 0:
            mlflow.log_metrics({
                "loss_total":   loss_total.item(),
                "loss_data_H":  data_loss_H.item(),
                "loss_data_F":  data_loss_F.item(),
                "loss_physics": physik_loss_val.item(),
                "alpha":        alpha.item(),
                "beta":         beta.item(),
                "gamma":        gamma.item(),
                "delta":        delta.item()
            }, step=ep)

        if ep % 500 == 0:
            print(f"Epoch {ep:5d} | "
                  f"loss={loss_total.item():.4f} | "
                  f"alpha={alpha.item():.3f} (1.0) | "
                  f"beta={beta.item():.3f} (0.1) | "
                  f"gamma={gamma.item():.3f} (1.5) | "
                  f"delta={delta.item():.4f} (0.075)")

    # Plot speichern
    t_dense = torch.linspace(0, 15, 300).reshape(-1, 1).to(device)
    with torch.no_grad():
        H_pred_dense, F_pred_dense = model(t_dense)

    t_np = t_dense.cpu().numpy()
    plt.figure(figsize=(12, 5))
    plt.plot(sol.t, sol.y[0], 'b-', label='Hasen (echt)', alpha=0.4)
    plt.scatter(sol.t, H_noise, color='blue', s=15)
    plt.plot(t_np, H_pred_dense.cpu().numpy(), 'b--', linewidth=2, label='PINN Hasen')
    plt.plot(sol.t, sol.y[1], 'r-', label='Füchse (echt)', alpha=0.4)
    plt.scatter(sol.t, F_noise, color='red', s=15)
    plt.plot(t_np, F_pred_dense.cpu().numpy(), 'r--', linewidth=2, label='PINN Füchse')
    plt.xlabel("Zeit t")
    plt.ylabel("Population")
    plt.title("Lotka-Volterra PINN — Inverses Problem")
    plt.legend()
    plt.grid(True)
    plt.savefig("lv_plot.png")
    mlflow.log_artifact("lv_plot.png")
    plt.close()

    mlflow.log_metrics({
        "final_alpha": alpha.item(),
        "final_beta":  beta.item(),
        "final_gamma": gamma.item(),
        "final_delta": delta.item()
    }, step=epochs)

print("Training fertig.")
print(f"alpha={alpha.item():.3f} | beta={beta.item():.3f} | gamma={gamma.item():.3f} | delta={delta.item():.4f}")

Epoch   500 | loss=137.5231 | alpha=1.360 (1.0) | beta=0.131 (0.1) | gamma=0.730 (1.5) | delta=0.0352 (0.075)
Epoch  1000 | loss=126.4813 | alpha=1.329 (1.0) | beta=0.127 (0.1) | gamma=0.879 (1.5) | delta=0.0422 (0.075)
Epoch  1500 | loss=103.7014 | alpha=1.234 (1.0) | beta=0.117 (0.1) | gamma=1.117 (1.5) | delta=0.0536 (0.075)
Epoch  2000 | loss=54.4914 | alpha=1.264 (1.0) | beta=0.119 (0.1) | gamma=1.129 (1.5) | delta=0.0538 (0.075)
Epoch  2500 | loss=32.6922 | alpha=1.094 (1.0) | beta=0.101 (0.1) | gamma=1.306 (1.5) | delta=0.0630 (0.075)
Epoch  3000 | loss=15.8442 | alpha=0.965 (1.0) | beta=0.092 (0.1) | gamma=1.519 (1.5) | delta=0.0742 (0.075)
Epoch  3500 | loss=6.5690 | alpha=0.991 (1.0) | beta=0.096 (0.1) | gamma=1.447 (1.5) | delta=0.0720 (0.075)
Epoch  4000 | loss=3.1241 | alpha=1.013 (1.0) | beta=0.100 (0.1) | gamma=1.408 (1.5) | delta=0.0707 (0.075)
Epoch  4500 | loss=1.6726 | alpha=1.018 (1.0) | beta=0.101 (0.1) | gamma=1.415 (1.5) | delta=0.0713 (0.075)
Epoch  5000 | loss=